# E05 - PHOBERT ABLATION: `MAX_LEN` (128 VS 256), `LR` VÀ CÁCH CHIA DEV

**Câu hỏi (từ bài review mục 3 & 10):**
1. NB02 đo được `p99 = 119 BPE subwords`. Giảm `MAX_LEN` từ `256` xuống `128` mất bao nhiêu % câu bị cắt, nhưng train nhanh hơn bao nhiêu và Dev F1 có đổi không?
2. Dùng 1 tập `dev_clean` thống nhất cho early stopping với các mức `LR` (`1e-5`, `2e-5`, `3e-5`).

**Tương thích:** chạy được trên **Local**, **Google Colab**, và **Kaggle GPU**.

In [ ]:
import os, sys, glob, zipfile, subprocess

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle/working')

def _find_root():
    cwd = os.getcwd()
    cand = os.path.abspath('..') if os.path.basename(cwd) in ('notebooks', 'experiments') else cwd
    if os.path.exists(os.path.join(cand, 'src', 'modeling.py')):
        return cand, cand

    if IN_COLAB:
        for base in [cwd, '/content']:
            if os.path.exists(os.path.join(base, 'src', 'modeling.py')):
                return base, base
        for z in glob.glob('/content/*src_and_data*.zip'):
            with zipfile.ZipFile(z, 'r') as zf:
                zf.extractall('/content')
            if os.path.exists('/content/src/modeling.py'):
                return '/content', '/content'

    if IN_KAGGLE:
        work = '/kaggle/working'
        if os.path.exists(os.path.join(work, 'src', 'modeling.py')):
            return work, work
        for p in glob.glob('/kaggle/input/**/src/modeling.py', recursive=True):
            data_root = os.path.dirname(os.path.dirname(p))
            return data_root, work
        for z in glob.glob('/kaggle/input/**/*src_and_data*.zip', recursive=True):
            dest = os.path.join(work, 'project')
            os.makedirs(dest, exist_ok=True)
            with zipfile.ZipFile(z, 'r') as zf:
                zf.extractall(dest)
            if os.path.exists(os.path.join(dest, 'src', 'modeling.py')):
                return dest, dest

    raise FileNotFoundError('Khong tim thay src/modeling.py! Hay chay trong repo hoac upload/attach src_and_data.zip.')

ROOT, WORK_DIR = _find_root()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
PROC_DIR = os.path.join(ROOT, 'data', 'processed')
os.environ['EXP_LOG_PATH'] = os.path.join(WORK_DIR, 'reports', 'experiments_log.csv')

for pkg in ['transformers', 'accelerate', 'scikit-learn']:
    mod = 'sklearn' if pkg == 'scikit-learn' else pkg
    try:
        __import__(mod)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print(f'Env: colab={IN_COLAB}, kaggle={IN_KAGGLE} | ROOT={ROOT} | WORK_DIR={WORK_DIR}')

Env: colab=True, kaggle=False | ROOT=/content | WORK_DIR=/content


In [ ]:
import time
import numpy as np
import pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score
from src.modeling import PII_TOKENS, ViSFDDataset, PhoBertMultiTask, dynamic_collate_fn, init_semantic_pii_embeddings
from src.exp_log import log_experiment

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


In [ ]:
df_train = pd.read_csv(os.path.join(PROC_DIR, 'train_clean.csv'))
df_dev = pd.read_csv(os.path.join(PROC_DIR, 'dev_clean.csv'))
tok = AutoTokenizer.from_pretrained('vinai/phobert-base-v2')
tok.add_special_tokens({'additional_special_tokens': PII_TOKENS})

bpe_dev = df_dev['sentence_seg'].fillna('').apply(lambda s: len(tok.encode(str(s), truncation=False)))
for L in [128, 192, 256]:
    print(f'MAX_LEN={L}: dev <= {L} = {(bpe_dev <= L).mean():.2%} (max={bpe_dev.max()})')

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

MAX_LEN=128: dev <= 128 = 99.55% (max=245)
MAX_LEN=192: dev <= 192 = 99.91% (max=245)
MAX_LEN=256: dev <= 256 = 100.00% (max=245)


## 1. THỬ NGHIỆM `MAX_LEN` VÀ `LR` TRÊN CÙNG TẬP DEV

In [ ]:
asp_cols = [f'asp_{i}' for i in range(5)]
pin = (device.type == 'cuda')

def run_phobert(max_len=128, lr=2e-5, epochs=4, batch_size=32):
    torch.manual_seed(42)
    np.random.seed(42)
    tr_ds = ViSFDDataset(df_train['sentence_seg'].fillna(''), tok, df_train['sentiment'], df_train[asp_cols].values, max_len=max_len)
    dv_ds = ViSFDDataset(df_dev['sentence_seg'].fillna(''), tok, df_dev['sentiment'], df_dev[asp_cols].values, max_len=max_len)
    tr_dl = DataLoader(tr_ds, batch_size=batch_size, shuffle=True, pin_memory=pin, collate_fn=lambda b: dynamic_collate_fn(b, tok.pad_token_id))
    dv_dl = DataLoader(dv_ds, batch_size=64, shuffle=False, pin_memory=pin, collate_fn=lambda b: dynamic_collate_fn(b, tok.pad_token_id))

    model = PhoBertMultiTask('vinai/phobert-base-v2', dropout=0.3).to(device)
    init_semantic_pii_embeddings(model.encoder, tok)
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * len(tr_dl) * epochs), len(tr_dl) * epochs)
    loss_s = nn.CrossEntropyLoss(label_smoothing=0.05)
    loss_a = nn.BCEWithLogitsLoss()

    best_mean, best_info = -1.0, {}
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        for b in tr_dl:
            opt.zero_grad()
            ls, la = model(b['input_ids'].to(device), b['attention_mask'].to(device))
            loss = loss_s(ls, b['sentiment'].to(device)) + 0.9 * loss_a(la, b['aspects'].to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
        model.eval()
        ps, pa = [], []
        with torch.no_grad():
            for b in dv_dl:
                ls, la = model(b['input_ids'].to(device), b['attention_mask'].to(device))
                ps.append(ls.argmax(-1).cpu().numpy())
                pa.append((la >= 0).int().cpu().numpy())
        sf1 = f1_score(df_dev['sentiment'], np.concatenate(ps), average='macro')
        af1 = f1_score(df_dev[asp_cols].values, np.vstack(pa), average='macro')
        mf1 = (sf1 + af1) / 2.0
        if mf1 > best_mean:
            best_mean = mf1
            best_info = {'best_ep': ep, 'sent_f1': round(sf1, 4), 'asp_f1': round(af1, 4), 'mean_f1': round(mf1, 4)}
    sec = round(time.time() - t0, 1)
    run_name = f'maxlen_{max_len}_lr_{lr}'
    log_experiment('E05', run_name, 'PhoBERT MAX_LEN & LR ablation',
                   {'max_len': max_len, 'lr': lr, 'epochs': epochs},
                   dev_f1=best_mean, notes=f'{best_info}, sec={sec}')
    return {'run': run_name, **best_info, 'sec': sec}

grid = [(128, 2e-5), (256, 2e-5), (128, 3e-5), (128, 1e-5)]
if device.type == 'cuda':
    results = [run_phobert(ml, lr) for ml, lr in grid]
    df_out = pd.DataFrame(results)
    print(df_out)
    print('Log saved to:', os.environ['EXP_LOG_PATH'])
else:
    print('Khong thay GPU CUDA -> hay bat GPU tren Colab/Kaggle de chay cell nay.')

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  540MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


model.safetensors: reconstructing file:   0%|          |  0.00B /  540MB            

model.safetensors: downloading bytes:           |  0.00B            

... (3 outputs left out)

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


                   run  best_ep  sent_f1  asp_f1  mean_f1    sec
0  maxlen_128_lr_2e-05        4   0.8048  0.8916   0.8482  633.7
1  maxlen_256_lr_2e-05        3   0.7989  0.8899   0.8444  680.5
2  maxlen_128_lr_3e-05        4   0.8102  0.9091   0.8597  647.2
3  maxlen_128_lr_1e-05        4   0.7985  0.8151   0.8068  647.1
Log saved to: /content/reports/experiments_log.csv


## 2. KẾT LUẬN

- **`MAX_LEN=128` vs `256` (cùng `lr=2e-5`):** `MAX_LEN=128` giữ được `99.55%` số câu dev, chạy nhanh hơn (`633.7s` vs `680.5s`) và cho Dev Mean F1 cao hơn (`0.8482` vs `0.8444`).
- **Learning rate (`MAX_LEN=128`, 4 epochs):** `lr=3e-5` đạt kết quả tốt nhất với **`mean_f1 = 0.8597`** (`sent_f1 = 0.8102`, `asp_f1 = 0.9091` tại epoch 4), vượt xa `2e-5` (`0.8482`) và `1e-5` (`0.8068`).
- **Gợi ý cho `04_phobert_multitask.ipynb`:** giảm `MAX_LEN` xuống `128` và tăng `LR` lên `3e-5` (giữ `EPOCHS=6` + early stopping trên 1 tập dev thống nhất).